# qwen2.5_1.5b GRPO evaluation


In [ ]:
import os, subprocess, sys
from pathlib import Path

LAUNCH_DIR = Path.cwd().resolve()
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv>=1,<2"], check=True)
from dotenv import load_dotenv

ENV_FILE = Path(os.environ.get("CRASHDIAG_ENV_FILE", LAUNCH_DIR / ".env")).expanduser()
if not ENV_FILE.is_absolute():
    ENV_FILE = (LAUNCH_DIR / ENV_FILE).resolve()
if ENV_FILE.is_file():
    load_dotenv(ENV_FILE, override=True)

try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    UserSecretsClient = None

KAGGLE_SECRET_ALIASES = {
    "HF_TOKEN": "HF_TOKEN",
    "CRASHDIAG_DATASET_RUN_ID": "CRASHDIAG_DATASET_RUN_ID",
    "DATASET_RUN_ID": "CRASHDIAG_DATASET_RUN_ID",
    "CRASHDIAG_SANDBOX_URL": "CRASHDIAG_SANDBOX_URL",
    "CRASHDIAG_API_TOKEN": "CRASHDIAG_API_TOKEN",
    "CRASHDIAG_SANDBOX_TOKEN": "CRASHDIAG_API_TOKEN",
    "CRASHDIAG_SOURCE_COMMIT": "CRASHDIAG_SOURCE_COMMIT",
    "SOURCE_COMMIT": "CRASHDIAG_SOURCE_COMMIT",
    "CRASHDIAG_GRPO_RUN_ID": "CRASHDIAG_GRPO_RUN_ID",
    "GRPO_RUN_ID": "CRASHDIAG_GRPO_RUN_ID",
    "CRASHDIAG_GRPO_EVAL_RUN_ID": "CRASHDIAG_GRPO_EVAL_RUN_ID",

}
loaded_kaggle_secrets = []
kaggle_secret_errors = {}
if UserSecretsClient is not None:
    secrets = UserSecretsClient()
    for secret_name, env_name in KAGGLE_SECRET_ALIASES.items():
        if os.environ.get(env_name):
            continue
        try:
            value = secrets.get_secret(secret_name)
        except Exception as exc:
            kaggle_secret_errors[secret_name] = f"{type(exc).__name__}: {exc}"
            continue
        if value:
            os.environ[env_name] = value
            loaded_kaggle_secrets.append(secret_name)
print("loaded Kaggle secret names:", loaded_kaggle_secrets or "none")

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

REPO_URL = os.environ.get("CRASHDIAG_REPO_URL", "https://github.com/Indium-AI-Labs/CrashDiag.git")
SOURCE_COMMIT = os.environ.get("CRASHDIAG_SOURCE_COMMIT", "main")
WORKDIR = Path(os.environ.get("CRASHDIAG_WORKDIR", LAUNCH_DIR / "CrashDiag-runtime")).expanduser().resolve()
if (WORKDIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
elif WORKDIR.exists() and any(WORKDIR.iterdir()):
    raise RuntimeError(f"CRASHDIAG_WORKDIR exists and is not a Git checkout: {WORKDIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "-C", str(WORKDIR), "checkout", SOURCE_COMMIT], check=True)
os.chdir(WORKDIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "bitsandbytes"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
print(f"env_file={ENV_FILE if ENV_FILE.is_file() else 'not present (using runtime/Kaggle secrets)'}")
print("checked_out_source_commit=" + subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import os

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_SLUG = "qwen2.5_1.5b"
BUCKET_ID = "devaanshpa/CrashDiag"
DATASET_RUN_ID = os.environ.get("CRASHDIAG_DATASET_RUN_ID", "").strip()
def ist_run_id(stage):
    return datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y%m%dT%H%M%SIST") + f"-{MODEL_SLUG}-{stage}"
if not DATASET_RUN_ID:
    raise RuntimeError("Set CRASHDIAG_DATASET_RUN_ID to the fresh dataset-generation run ID.")
print(f"base_model={BASE_MODEL}")
print(f"dataset_run_id={DATASET_RUN_ID}")
GRPO_RUN_ID = os.environ.get("CRASHDIAG_GRPO_RUN_ID", "").strip()
if not GRPO_RUN_ID:
    detail = kaggle_secret_errors.get("CRASHDIAG_GRPO_RUN_ID") or kaggle_secret_errors.get("GRPO_RUN_ID")
    raise RuntimeError("Missing CRASHDIAG_GRPO_RUN_ID. Add and enable that Kaggle Secret "
                       "(or GRPO_RUN_ID), then rerun from the first cell. "
                       f"Kaggle response: {detail or 'secret was not returned'}")
GRPO_EVAL_RUN_ID = os.environ.get("CRASHDIAG_GRPO_EVAL_RUN_ID") or ist_run_id("grpo-eval")
print(f"grpo_run_id={GRPO_RUN_ID}")
print(f"grpo_eval_run_id={GRPO_EVAL_RUN_ID}")


In [ ]:
from pathlib import Path
from training.artifacts import ArtifactConfig, ArtifactUploader

CURRICULUM = os.environ.get("CRASHDIAG_CURRICULUM", "hard-v4").strip().lower()
HARD = CURRICULUM in ("hard-v3", "hard-v4")
TRAIN_FILE = "grpo_hard_train.jsonl" if HARD else "grpo_train.jsonl"
EVAL_FILE = "grpo_hard_eval.jsonl" if HARD else "grpo_eval.jsonl"
DATASET_DIR = Path("artifacts/datasets")
GRPO_DIR = Path("artifacts/grpo")
token = os.environ["HF_TOKEN"]
ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=DATASET_RUN_ID, token=token)).download_stage("datasets", DATASET_DIR)
ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=GRPO_RUN_ID, token=token)).download_stage("grpo", GRPO_DIR)
assert (DATASET_DIR / EVAL_FILE).is_file(), f"dataset stage missing {EVAL_FILE}; check CRASHDIAG_DATASET_RUN_ID={DATASET_RUN_ID}"
print(f"curriculum={CURRICULUM}")
print(f"eval_file={EVAL_FILE}")


In [ ]:
from training.evaluate_jsonl import main as evaluate_main

exit_code = evaluate_main([
    "--model", str(GRPO_DIR), "--dataset", str(DATASET_DIR / EVAL_FILE),
    "--output-dir", "outputs/grpo-eval", "--load-in-4bit", "--precision", "bf16",
    "--max-new-tokens", "64",
    "--sandbox-url", os.environ["CRASHDIAG_SANDBOX_URL"],
    "--artifact-bucket", BUCKET_ID, "--run-id", GRPO_EVAL_RUN_ID, "--artifact-stage", "grpo-eval",
    "--no-few-shot",
])
if exit_code: raise RuntimeError(f"GRPO evaluation failed: {exit_code}")


In [ ]:
from IPython.display import SVG, display

REPORTS_DIR = Path("outputs/grpo-eval") / "reports"
charts = sorted(REPORTS_DIR.glob("*.svg"))
if not charts:
    raise RuntimeError(f"No SVG charts were generated in {REPORTS_DIR}")
print(f"Uploaded reports: hf://buckets/{BUCKET_ID}/runs/{GRPO_EVAL_RUN_ID}/grpo-eval/reports")
for chart in charts:
    display(SVG(filename=str(chart)))
